# MarketMinds — Phase 3c extended: LSTM on Devyani's 5 additional windows

Devyani independently extended the linear regression/ARIMA/Random Forest/XGBoost evaluation from our original 3 windows to 8 (matching the historical periods from the marketing chart), and her `ml-lab.html` explicitly notes LSTM isn't represented yet. This notebook closes that gap for 5 of those windows -- everything except `dotcom`, which needs a separate data-pipeline decision first (her data goes back to 1998; ours starts 2000-01-03, and `dotcom` wants training data from before our data even begins).

Window boundaries below are reverse-engineered from the `window_label` text in her `assets/js/data.js` (e.g. `'Bull run (trained thru Mar 2003, tested 1 Apr 2003-8 Jan 2008)'`), so results here should slot directly next to her existing Random Forest/XGBoost numbers for the same windows.

Everything else is identical to `05_lstm_model.ipynb`: same features, same target, same architecture, same train-once-per-window design, same full 20-asset basket. Reuses `src/forecasting.py` and `src/sequence_model.py` unchanged -- no new engine code, just new window definitions.

**Runtime note**: unlike linear regression/ARIMA (daily refit) or Random Forest/XGBoost (refit every 20 days), LSTM trains once per window regardless of the window's length -- so `bullrun` being a ~4.75-year test period doesn't multiply training cost the way it would for the other models, it just means more (cheap) inference steps. Expect this to be in the same ballpark as `05_lstm_model.ipynb`'s original runtime, not dramatically longer.

**Output**: same artifacts as before (models/ + results CSV), plus a final step that formats everything into the exact JS object shape her `ml-lab.html` expects, ready to hand back for integration -- not written directly into her repo from here, since that's a deliberate manual-merge step, not an automated overwrite.

**Note on execution**: same as always — you run every cell, I don't execute anything. Paste back any errors.

## Setup

In [28]:
import json
import pickle
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
torch.manual_seed(42)

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.append(str(PROJECT_ROOT))

from src.forecasting import FEATURE_COLS, build_feature_target_table, compute_eval_metrics, safe_asset_name
from src.sequence_model import build_sequences, train_eval_lstm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print('Using device:', device)

master = pq.read_table(PROCESSED_DIR / 'marketminds_master.parquet').to_pandas()
symbol_metadata = pq.read_table(PROCESSED_DIR / 'symbol_metadata.parquet').to_pandas()
print('Master shape:', master.shape)
print('Master date range:', master['date'].min(), '->', master['date'].max())

Using device: mps
Master shape: (134646, 18)
Master date range: 2000-01-03 00:00:00 -> 2026-08-11 00:00:00


## 1. The 5 new windows, plus basket definition

Boundaries reverse-engineered from her `window_label` strings; `train_end` set to the day before each `test_start`, matching the convention used everywhere else in this project.

In [29]:
ASSETS = sorted(t for t in master['ticker'].unique() if t != '^INDIAVIX')

NEW_WINDOWS = {
    'bullrun': {'train_end': '2003-03-31', 'test_start': '2003-04-01', 'test_end': '2008-01-08'},
    'recovery0910': {'train_end': '2009-03-08', 'test_start': '2009-03-09', 'test_end': '2010-11-05'},
    'demon2016': {'train_end': '2016-11-07', 'test_start': '2016-11-08', 'test_end': '2016-12-26'},
    'postcovid': {'train_end': '2020-03-22', 'test_start': '2020-03-23', 'test_end': '2021-10-18'},
    'correction2425': {'train_end': '2024-09-26', 'test_start': '2024-09-27', 'test_end': '2025-02-01'},
}

# bullrun is ~4.75 years (1246 trading days) with ~558% price growth over the window -- far
# longer/more-trending than any other window here. A single scaler/model fit once at the start
# goes badly out-of-distribution by the end (this is what produced the 17-23% MAE blowup on the
# first run). Refitting every ~6 months keeps the model/scaler current with the price level.
# Every other window is short enough that train-once (the default) is fine, same as before.
REFIT_EVERY = {
    'bullrun': 125,
}

MIN_TRAIN_ROWS = 250

print(len(ASSETS), 'assets')
for name, cfg in NEW_WINDOWS.items():
    print(name, '->', cfg, ', refit_every =', REFIT_EVERY.get(name))

20 assets
bullrun -> {'train_end': '2003-03-31', 'test_start': '2003-04-01', 'test_end': '2008-01-08'} , refit_every = 125
recovery0910 -> {'train_end': '2009-03-08', 'test_start': '2009-03-09', 'test_end': '2010-11-05'} , refit_every = None
demon2016 -> {'train_end': '2016-11-07', 'test_start': '2016-11-08', 'test_end': '2016-12-26'} , refit_every = None
postcovid -> {'train_end': '2020-03-22', 'test_start': '2020-03-23', 'test_end': '2021-10-18'} , refit_every = None
correction2425 -> {'train_end': '2024-09-26', 'test_start': '2024-09-27', 'test_end': '2025-02-01'} , refit_every = None


## 2. Train, evaluate, and save LSTM across the basket for each new window

Same save convention as `05_lstm_model.ipynb`: one `.pt` + `.json` per (asset, window).

In [30]:
def save_model(model_bundle, ticker, window_name, cfg, metrics, info):
    asset_name = safe_asset_name(ticker)
    base = 'lstm_' + asset_name + '_' + window_name
    model_path = MODELS_DIR / (base + '.pt')
    meta_path = MODELS_DIR / (base + '.json')

    torch.save(model_bundle, model_path)

    metadata = {
        'asset': ticker,
        'model_type': 'lstm',
        'window': window_name,
        'train_end': cfg['train_end'],
        'test_start': cfg['test_start'],
        'test_end': cfg['test_end'],
        'seq_len': model_bundle['seq_len'],
        'hidden_size': model_bundle['hidden_size'],
        'num_layers': model_bundle['num_layers'],
        'cell_type': model_bundle['cell_type'],
        'training_time_seconds': info['training_time_seconds'],
        'epochs_trained': info['epochs_trained'],
        'best_val_loss': info['best_val_loss'],
        'metrics': metrics,
        'date_trained': datetime.now().isoformat(),
    }
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)

In [31]:
SEQ_LEN = 30
HIDDEN_SIZE = 32
NUM_LAYERS = 1
CELL_TYPE = 'lstm'

# PyTorch's MPS backend isn't fully deterministic even with a fixed seed -- re-running the same
# window can land on a meaningfully different trained model (this showed up as postcovid flipping
# between "not significant" and "significantly worse" across two identical runs). Training
# N_REPEATS independently-seeded models per (asset, window) and averaging their predictions before
# computing metrics is standard ensemble variance-reduction, and directly fixes this.
N_REPEATS = 5

results = []
start_time = time.time()

for i, ticker in enumerate(ASSETS, 1):
    asset_df = master[master['ticker'] == ticker]
    feature_table = build_feature_target_table(asset_df)
    X, y, seq_dates = build_sequences(feature_table, SEQ_LEN)
    print(f'[{i}/{len(ASSETS)}] {ticker} -- {len(X)} sequences total, elapsed {time.time() - start_time:.0f}s')

    for window_name, cfg in NEW_WINDOWS.items():
        train_count = (seq_dates <= pd.Timestamp(cfg['train_end'])).sum()
        test_count = ((seq_dates >= pd.Timestamp(cfg['test_start'])) & (seq_dates <= pd.Timestamp(cfg['test_end']))).sum()

        if train_count < MIN_TRAIN_ROWS or test_count == 0:
            print(f'    {window_name}: skipped (train_count={train_count}, test_count={test_count})')
            results.append({
                'ticker': ticker, 'window': window_name, 'model': 'lstm',
                'status': 'skipped_insufficient_history',
                'mae_pct': np.nan, 'rmse_pct': np.nan, 'directional_accuracy_pct': np.nan,
                'dir_acc_ci_low': np.nan, 'dir_acc_ci_high': np.nan,
                'p_value_vs_random': np.nan, 'significant_vs_random': False,
                'significant_better_than_random': False, 'significant_worse_than_random': False,
                'n_predictions': int(train_count),
            })
            continue

        window_start = time.time()
        pred_runs = []
        actual, model_bundle, info = None, None, None
        for repeat in range(N_REPEATS):
            torch.manual_seed(42 + repeat)
            predicted_r, actual_r, model_bundle_r, info_r = train_eval_lstm(
                X, y, seq_dates, cfg, device,
                hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, cell_type=CELL_TYPE,
                refit_every=REFIT_EVERY.get(window_name),
            )
            pred_runs.append(predicted_r)
            actual = actual_r
            model_bundle, info = model_bundle_r, info_r  # keep the last repeat's model as the saved artifact
        predicted = sum(pred_runs) / len(pred_runs)
        window_time = time.time() - window_start

        metrics = compute_eval_metrics(actual, predicted)
        results.append({'ticker': ticker, 'window': window_name, 'model': 'lstm', 'status': 'ok', **metrics})
        save_model(model_bundle, ticker, window_name, cfg, metrics, info)

        print(f'    {window_name}: {window_time:.1f}s for {N_REPEATS} runs, '
              f'dir_acc={metrics["directional_accuracy_pct"]}%, n={metrics["n_predictions"]}')

print(f'\nDone in {time.time() - start_time:.0f}s -- {len(results)} rows')
lstm_extended_df = pd.DataFrame(results)

[1/20] DRREDDY.NS -- 6709 sequences total, elapsed 0s
    bullrun: 36.8s for 5 runs, dir_acc=51.4%, n=1246
    recovery0910: 6.5s for 5 runs, dir_acc=36.3%, n=435
    demon2016: 7.8s for 5 runs, dir_acc=48.6%, n=35
    postcovid: 7.8s for 5 runs, dir_acc=49.1%, n=411
    correction2425: 10.8s for 5 runs, dir_acc=54.9%, n=91
[2/20] GOLDBEES.NS -- 4360 sequences total, elapsed 70s
    bullrun: skipped (train_count=0, test_count=0)
    recovery0910: skipped (train_count=0, test_count=253)
    demon2016: 5.1s for 5 runs, dir_acc=37.1%, n=35
    postcovid: 7.5s for 5 runs, dir_acc=47.4%, n=411
    correction2425: 7.8s for 5 runs, dir_acc=57.1%, n=91
[3/20] HDFCBANK.NS -- 6709 sequences total, elapsed 90s
    bullrun: 31.5s for 5 runs, dir_acc=50.1%, n=1246
    recovery0910: 5.3s for 5 runs, dir_acc=49.4%, n=435
    demon2016: 8.0s for 5 runs, dir_acc=62.9%, n=35
    postcovid: 11.5s for 5 runs, dir_acc=49.9%, n=411
    correction2425: 6.1s for 5 runs, dir_acc=54.9%, n=91
[4/20] HINDUNILVR.N

## 3. Merge sector metadata, save detailed results

In [32]:
lstm_extended_df = lstm_extended_df.merge(symbol_metadata, on='ticker', how='left')
cols = ['ticker', 'category', 'window', 'model', 'status', 'mae_pct', 'rmse_pct',
        'directional_accuracy_pct', 'dir_acc_ci_low', 'dir_acc_ci_high',
        'p_value_vs_random', 'significant_vs_random',
        'significant_better_than_random', 'significant_worse_than_random', 'n_predictions']
lstm_extended_df = lstm_extended_df[cols]

out_path = RESULTS_DIR / 'lstm_extended_windows_validation.csv'
lstm_extended_df.to_csv(out_path, index=False)
print('Saved', len(lstm_extended_df), 'rows to', out_path)
lstm_extended_df.head(10)

Saved 100 rows to /Users/palakjagtap/Documents/Market_minds/results/lstm_extended_windows_validation.csv


,ticker,category,window,model,status,mae_pct,rmse_pct,directional_accuracy_pct,dir_acc_ci_low,dir_acc_ci_high,p_value_vs_random,significant_vs_random,significant_better_than_random,significant_worse_than_random,n_predictions
0,DRREDDY.NS,Pharma,bullrun,lstm,ok,4.400,13.467,51.4,48.6,54.1,0.3354,False,False,False,1246
1,DRREDDY.NS,Pharma,recovery0910,lstm,ok,9.678,11.051,36.3,31.9,40.9,0.0000,True,False,True,435
2,DRREDDY.NS,Pharma,demon2016,lstm,ok,2.225,2.675,48.6,33.0,64.4,0.8658,False,False,False,35
3,DRREDDY.NS,Pharma,postcovid,lstm,ok,4.541,5.708,49.1,44.3,54.0,0.7299,False,False,False,411
4,DRREDDY.NS,Pharma,correction2425,lstm,ok,2.765,3.456,54.9,44.7,64.8,0.3454,False,False,False,91
5,GOLDBEES.NS,Gold,bullrun,lstm,skipped_insufficient_history,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0
6,GOLDBEES.NS,Gold,recovery0910,lstm,skipped_insufficient_history,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0
7,GOLDBEES.NS,Gold,demon2016,lstm,ok,2.067,2.328,37.1,23.2,53.7,0.1282,False,False,False,35
8,GOLDBEES.NS,Gold,postcovid,lstm,ok,4.025,4.754,47.4,42.7,52.3,0.3003,False,False,False,411
9,GOLDBEES.NS,Gold,correction2425,lstm,ok,1.845,2.276,57.1,46.9,66.8,0.1730,False,False,False,91


## 4. Format for `assets/js/data.js`

Builds the exact JS snippets her `ml-lab.html` expects: per-window `lstm` entries under `MM.ML_METRICS.bsesn.<window>` (matching how `linear_regression`/`arima` only appear for the index), and `{model: 'lstm', ...}` rows to append to `basketSignificance` (matching how `random_forest`/`xgboost` appear across the full basket). Printed here for review, not written directly into her repo -- merging into `data.js` is a deliberate manual step.

In [33]:
WINDOW_LABEL_SUFFIX = {
    'bullrun': 'Bull run (trained thru Mar 2003, tested 1 Apr 2003\u20138 Jan 2008)',
    'recovery0910': 'Recovery rally (trained thru Mar 2009, tested 9 Mar 2009\u20135 Nov 2010)',
    'demon2016': 'Demonetization (trained thru 7 Nov 2016, tested 8 Nov\u201326 Dec 2016)',
    'postcovid': 'Post-COVID rally (trained thru 22 Mar 2020, tested 23 Mar 2020\u201318 Oct 2021)',
    'correction2425': '2024\u201325 correction (trained thru 26 Sep 2024, tested 27 Sep 2024\u20131 Feb 2025)',
}

def js_bool(v):
    return 'true' if v else 'false'

print('// --- paste into MM.ML_METRICS.bsesn.<window>.lstm for each window ---\n')
for window_name in NEW_WINDOWS:
    row = lstm_extended_df[(lstm_extended_df['ticker'] == '^BSESN') & (lstm_extended_df['window'] == window_name)]
    if row.empty or row.iloc[0]['status'] != 'ok':
        print(f'// {window_name}: no SENSEX result (skipped or missing)')
        continue
    r = row.iloc[0]
    ci = f"[{r['dir_acc_ci_low']}, {r['dir_acc_ci_high']}]" if pd.notna(r['dir_acc_ci_low']) else 'null'
    print(f"// {window_name}")
    print(f"lstm: {{ mae_pct: {r['mae_pct']}, rmse_pct: {r['rmse_pct']}, dir_acc_pct: {r['directional_accuracy_pct']}, "
          f"ci: {ci}, sig_better: {js_bool(r['significant_better_than_random'])}, "
          f"sig_worse: {js_bool(r['significant_worse_than_random'])}, p: {r['p_value_vs_random']}, n: {int(r['n_predictions'])} }},\n")

print()
print('// --- append to MM.ML_METRICS.basketSignificance ---\n')
for window_name in NEW_WINDOWS:
    ok_rows = lstm_extended_df[(lstm_extended_df['window'] == window_name) & (lstm_extended_df['status'] == 'ok')]
    total = len(ok_rows)
    if total == 0:
        print(f'// {window_name}: no results')
        continue
    sig_better = int(ok_rows['significant_better_than_random'].sum())
    sig_worse = int(ok_rows['significant_worse_than_random'].sum())
    pct_better = round(sig_better / total * 100, 1)
    pct_worse = round(sig_worse / total * 100, 1)
    print(f"{{ model: 'lstm', window: '{window_name}', total: {total}, sigBetter: {sig_better}, sigWorse: {sig_worse}, pctBetter: {pct_better}, pctWorse: {pct_worse} }},")

// --- paste into MM.ML_METRICS.bsesn.<window>.lstm for each window ---

// bullrun
lstm: { mae_pct: 3.445, rmse_pct: 4.37, dir_acc_pct: 42.0, ci: [39.3, 44.7], sig_better: false, sig_worse: true, p: 0.0, n: 1246 },

// recovery0910
lstm: { mae_pct: 2.611, rmse_pct: 3.761, dir_acc_pct: 59.3, ci: [54.6, 63.8], sig_better: true, sig_worse: false, p: 0.0001, n: 435 },

// demon2016
lstm: { mae_pct: 1.856, rmse_pct: 2.293, dir_acc_pct: 37.1, ci: [23.2, 53.7], sig_better: false, sig_worse: false, p: 0.1282, n: 35 },

// postcovid
lstm: { mae_pct: 2.178, rmse_pct: 2.934, dir_acc_pct: 50.1, ci: [45.3, 54.9], sig_better: false, sig_worse: false, p: 0.9607, n: 411 },

// correction2425
lstm: { mae_pct: 2.318, rmse_pct: 2.969, dir_acc_pct: 59.3, ci: [49.1, 68.9], sig_better: false, sig_worse: false, p: 0.0747, n: 91 },


// --- append to MM.ML_METRICS.basketSignificance ---

{ model: 'lstm', window: 'bullrun', total: 11, sigBetter: 0, sigWorse: 2, pctBetter: 0.0, pctWorse: 18.2 },
{ model: 'lstm

## Summary

What this notebook produced:
- `results/lstm_extended_windows_validation.csv` -- detailed per-asset LSTM results for the 5 new windows, same format as the earlier LSTM results file
- New `.pt` + `.json` model artifacts in `models/`, one per (asset, window)
- Step 4's printed JS snippets -- copy these back so they can be merged into `assets/js/data.js` and `ml-lab.html` (adding LSTM as a 5th model card, a 5th column in each window's table, and new bars in the significance chart)

`dotcom` is still not covered -- that needs a decision on extending the data pipeline back to 1998 (or an alternative) before it can be trained the same way.